# NVIDIA 3D Body Pose - Python Notebook

This notebook demonstrates how to use the NVIDIA 3D Body Pose NIM with Python.
The service estimates the 3D body pose of every tracked person in a video, over bidirectional gRPC streaming.

## Overview

The client streams a **compressed video file** as raw byte chunks, together with a **tracked bounding-box
annotation**, to the server. The server decodes the video frame by frame, runs 3D body pose estimation, and
streams back the per-frame pose for each tracked body:

- **2D keypoints** - image-space joint locations
- **3D keypoints** - camera-space joint locations
- **Joint rotations** - per-joint quaternions
- **Rest pose** and **root pose** - skeleton rest state and root transform

The service does **not** run body detection; you supply the tracked boxes per frame. See the package
`README.md` for the annotation file format and the command-line reference.

## Installation

### Requirements:
- Python 3.10 to 3.12
- pip package manager
- Dependencies from `../requirements.txt` (gRPC, OpenCV, and related packages)

Run the cell below to install all required packages:

In [ ]:
%pip install -r ../requirements.txt

## Service Configuration

Configure the connection to your NVIDIA 3D Body Pose service. To run the NIM, follow the
[documentation](https://docs.nvidia.com/nim/maxine/body-pose/latest/index.html).

In [ ]:
import os
import sys
import pathlib

SCRIPT_PATH = str(pathlib.Path().resolve())
sys.path.append(os.path.join(SCRIPT_PATH, "../scripts"))

SERVICE_HOST = "localhost"
SERVICE_PORT = 8001
SERVICE_TARGET = f"{SERVICE_HOST}:{SERVICE_PORT}"

print(f"Service target configured: {SERVICE_TARGET}")

## Import Libraries

The notebook reuses the functions of `scripts/body_pose.py`, the command-line client, instead of
reimplementing the gRPC streaming and the skeleton overlay.

In [ ]:
import grpc

from body_pose import (
    body_pose_pb2_grpc,
    generate_request_for_inference,
    process_responses,
    read_tracked_bboxes,
    write_overlay_video,
)
from config import BodyPoseClientConfig

print("All libraries imported successfully!")

## Estimate Poses

Run pose estimation on the bundled sample clip and render the skeleton overlay. The overlay reproduces the
AR SDK `BodyPose3DApp` sample: 2D keypoints in green, 3D keypoints reprojected with a pinhole camera in red,
and per-track colored boxes with ID labels.

The server buffers a temporal window before it sends the first poses, so they arrive after a delay and then
one frame at a time.

In [ ]:
config = BodyPoseClientConfig(
    video_filepath="../assets/sample_video.mp4",  # Update with your video
    bbox_filepath="../assets/sample_bbox.txt",  # Update with your annotation
    output_filepath="body_pose_output.json",
    overlay_filepath="body_pose_overlay.mp4",
    draw_keypoints="both",
    focal_length=0.0,
    enable_contact=None,
    timeout=3600.0,
    client_session_id=None,
)
config.validate_config()
tracked_bboxes = read_tracked_bboxes(config.bbox_filepath)

with grpc.insecure_channel(SERVICE_TARGET) as channel:
    stub = body_pose_pb2_grpc.BodyPoseServiceStub(channel)
    responses = stub.EstimateBodyPose(
        generate_request_for_inference(config, tracked_bboxes), timeout=config.timeout
    )
    for key, value in responses.initial_metadata() or ():
        if key == "warning":
            print(f"Server warning: {value}")
    poses_by_frame, focal_length = process_responses(responses, config)

write_overlay_video(config, poses_by_frame, focal_length)

## Contact Correction

`enable_contact=True` turns on static-camera contact correction, which runs inverse kinematics and is
therefore significantly slower. The container boots with it **off**.

This is a **server-wide** setting, fixed when the server starts (`BODY_POSE_ENABLE_CONTACT`). A request
asking for the other value is still served, with the server's setting, and the server says so in a
`warning` entry of the stream's initial metadata, printed above as `Server warning: ...`. Leave
`enable_contact=None` to accept whatever the server runs with.

## Pose Output Format

The output is JSON Lines: one `{"frame_id", "detections": [...]}` object per line, matching the AR SDK
sample app's `--out_json` shape. Per-joint arrays are all length **77** (the Nova-77 skeleton).

---

For more information, see the [NVIDIA 3D Body Pose NIM](https://docs.nvidia.com/nim/maxine/body-pose/latest/index.html) documentation.